# HUGO MANZANO 36231


In [1]:
!pip -q install sentence-transformers


In [2]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.datasets import fetch_20newsgroups


## Clases base

Solo creamos las clases con sus atributis

In [3]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


## Aqui hacemos la implementacion del vector store


In [4]:
class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        if not documents:
            return

        texts = [doc.text for doc in documents]

        # Convertimos cada documento en un vector. normalize_embeddings=True ayuda
        # porque después el producto punto equivale a similitud coseno
        new_embeddings = self.embedding_model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        self.documents.extend(documents)

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if self.embeddings is None or len(self.documents) == 0:
            return []

        query_embedding = self.embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]

        # Como los embeddings están normalizados  esto es cosine similarity
        scores = self.embeddings @ query_embedding

        top_indices = np.argsort(scores)[::-1][:top_k]

        return [
            SearchResult(score=float(scores[i]), document=self.documents[i])
            for i in top_indices
        ]


## Cargar Animal Fun Facts Dataset como documentos


In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

animal_url = "https://raw.githubusercontent.com/ekohrt/animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv"
animal_df = pd.read_csv(animal_url)

animal_df.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",NaN,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,NaN,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,NaN,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",NaN,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,NaN,/wiki/Aardvark


In [6]:
# text es el texto del documento.
# animal_name, source, media_link y wikipedia_link son metadatos.
animal_documents = []

for _, row in animal_df.iterrows():
    text = str(row["text"])

    metadata = {
        "animal_name": "" if pd.isna(row.get("animal_name")) else str(row.get("animal_name")),
        "source": "" if pd.isna(row.get("source")) else str(row.get("source")),
        "media_link": "" if pd.isna(row.get("media_link")) else str(row.get("media_link")),
        "wikipedia_link": "" if pd.isna(row.get("wikipedia_link")) else str(row.get("wikipedia_link")),
    }

    animal_documents.append(Document(text=text, metadata=metadata))

print(f"Documentos cargados: {len(animal_documents)}")
print(animal_documents[0].text)
print(animal_documents[0].metadata)


Documentos cargados: 7734
Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters"
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}


In [ ]:
animal_store = VectorStore(model)
animal_store.add_documents(animal_documents)


Batches:   0%|          | 0/242 [00:00<?, ?it/s]

## Consultas de ejemplo con VectorStore


In [ ]:
def print_results(results: list[SearchResult]):
    for i, result in enumerate(results, start=1):
        print(f"Resultado {i}")
        print(f"Score: {result.score:.4f}")
        print(f"Texto: {result.document.text}")
        print(f"Metadatos: {result.document.metadata}")
        print("-" * 100)


In [ ]:
animal_queries = [
    "animals that can camouflage",
    "facts about birds that fly long distances",
    "animals with strong sense of smell",
    "dangerous marine animals",
    "animals that live in deserts",
]

for query in animal_queries:
    print("=" * 120)
    print(f"Consulta: {query}")
    print_results(animal_store.search(query, top_k=3))


## Aqui filtraremos por metadata


In [ ]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

#agregamos documentos al vector store
    def add_documents(self, documents: list[Document]):
        if not documents:
            return

        texts = [doc.text for doc in documents]

        new_embeddings = self.embedding_model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        self.documents.extend(documents)

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

#con este metodo buscamos documentos similares a una consilta
    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        if self.embeddings is None or len(self.documents) == 0:
            return []

        if metadata_filter is None:
            candidate_indices = list(range(len(self.documents)))
        else:
            candidate_indices = []

            for i, doc in enumerate(self.documents):
                matches_filter = True

                for key, expected_value in metadata_filter.items():
                    actual_value = doc.metadata.get(key)

                    # Filtro simple por igualdad exacta en metadatos
                    if actual_value != expected_value:
                        matches_filter = False
                        break

                if matches_filter:
                    candidate_indices.append(i)

        if len(candidate_indices) == 0:
            return []

        query_embedding = self.embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]

        candidate_embeddings = self.embeddings[candidate_indices]
        scores = candidate_embeddings @ query_embedding

        order = np.argsort(scores)[::-1][:top_k]

        return [
            SearchResult(
                score=float(scores[position]),
                document=self.documents[candidate_indices[position]]
            )
            for position in order
        ]


## el dataset es 20 newgroup y pues este tiene cosas de noticias y mensajes que estan agrupados por categoria y de metadatos esta el category source e id


In [ ]:
categories = [
    "rec.autos",
    "sci.space",
    "comp.graphics",
    "talk.politics.misc",
]

news_data = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

news_documents = []

# aqui lo limite a 800 para que Colab corra más rápido jajaj, pero sigue siendo un dataset real
for i, text in enumerate(news_data.data[:800]):
    clean_text = " ".join(str(text).split())

    # Evitamos documentos vacíos o demasiado cortos
    if len(clean_text) < 80:
        continue

    category = news_data.target_names[news_data.target[i]]

    metadata = {
        "category": category,
        "source": "20 Newsgroups",
        "id": str(i),
    }

    news_documents.append(Document(text=clean_text, metadata=metadata))

print(f"Documentos cargados: {len(news_documents)}")
print(news_documents[0].text[:500])
print(news_documents[0].metadata)


In [ ]:
filtered_store = FilteredVectorStore(model)
filtered_store.add_documents(news_documents)


## Consultas de ejemplo con FilteredVectorStore


In [ ]:
filtered_queries = [
    ("car engine problems and performance", {"category": "rec.autos"}),
    ("NASA missions and satellites", {"category": "sci.space"}),
    ("rendering images and computer graphics", {"category": "comp.graphics"}),
    ("government policy and political debate", {"category": "talk.politics.misc"}),
    ("space shuttle technology", {"category": "sci.space"}),
]

for query, metadata_filter in filtered_queries:
    print("=" * 120)
    print(f"Consulta: {query}")
    print(f"Filtro: {metadata_filter}")
    print_results(filtered_store.search(query, top_k=3, metadata_filter=metadata_filter))


## Reflexiones personales

pues esta actividad me ayudó a entender como funciona bien el vector store y pues lo que me quedó claro es que en esencia guarda documentos, genera embeddings y compara una consulta contra esos vectores usando similitud y la parte mas importante fue ver que el texto se puede buscar por significado y no solo por palabras exactas, por eso una consulta puede encontrar documentos relacionados aunque no usen exactamente los mismos términos

También entendí la utilidad de los metadatos ya que lo veia mucho en videos y asi pero nunca entendi que eran jajaj. Finalmente, aprendí que la calidad de los resultados depende bastante del modelo de embeddings, de la limpieza del texto y de la calidad de los metadatos ya que si los metadatos estan incompletos o mal definidos, los filtros pueden hacer que no aparezcan documentos que ayuden o devolver resultados pues que no importen tanto
